# Section 5: Evaluation

*Duration: 15 minutes*

---

The agent loop ran. It selected tools, rewrote queries, and produced answers. But did it actually improve things?

This section scores the agent loop on two dimensions, not one. Answer correctness measures whether the final output matches the expected answer. Reasoning correctness measures whether the agent used the right tool for the right reason. A correct answer reached through wrong reasoning is luck, not reliability. A wrong answer reached through correct reasoning is a corpus gap, not an architecture failure.

Separating these two dimensions is the difference between knowing your system works and hoping it works.

## 5.1 Loading Results

Three result sets tell the full story:

1. **Passive RAG baseline** — the Escalation Lab's final evaluation (eval_results.json)
2. **Agent loop results** — the tool-augmented loop from Section 4 (agent_loop_results.json)

In [ ]:
import json

# Load the passive RAG baseline from the Escalation Lab
with open("../prebuilt/eval_results.json", "r", encoding="utf-8") as f:
    baseline_data = json.load(f)

baseline_results = baseline_data["results"]

# Load agent loop results from Section 4
with open("../prebuilt/agent_loop_results.json", "r", encoding="utf-8") as f:
    agent_data = json.load(f)

agent_results = agent_data["results"]

print(f"Baseline questions : {len(baseline_results)}")
print(f"Agent loop results : {len(agent_results)}")

## 5.2 Side-by-Side Comparison

The table below shows every question, the passive RAG result, and the agent loop result. Look for three patterns:

- **Preserved passes** — questions the passive pipeline got right that the agent loop also got right (no regression)
- **Recovered failures** — questions the passive pipeline got wrong that the agent loop fixed
- **Persistent failures** — questions neither architecture could answer correctly

In [ ]:
# Build a side-by-side comparison table
# The delta column highlights what changed between passive and agent

print(f"{'ID':<6} {'Category':<22} {'Passive':<12} {'Agent':<10} {'Delta'}")
print("=" * 75)

passive_pass_count = 0
agent_pass_count = 0
recovered = 0
regressed = 0

for br in baseline_results:
    # Find matching agent result
    ar = next((a for a in agent_results if a["id"] == br["id"]), None)
    
    passive_ok = br["classification"] == "pass"
    agent_ok = ar["agent_classification"] == "pass" if ar else False
    
    if passive_ok:
        passive_pass_count += 1
    if agent_ok:
        agent_pass_count += 1
    
    # Determine what changed
    if not passive_ok and agent_ok:
        delta = "RECOVERED"
        recovered += 1
    elif passive_ok and not agent_ok:
        delta = "REGRESSED"
        regressed += 1
    elif passive_ok and agent_ok:
        delta = "\u2014"
    else:
        delta = "still failing"
    
    passive_display = "pass" if passive_ok else "FAIL"
    agent_display = "pass" if agent_ok else "FAIL"
    
    print(f"{br['id']:<6} {br.get('category', ''):<22} {passive_display:<12} {agent_display:<10} {delta}")

print("=" * 75)
print(f"{'Total':<6} {'':<22} {passive_pass_count}/10{'':<7} {agent_pass_count}/10")
print(f"\nRecovered: {recovered}  |  Regressed: {regressed}  |  Net change: +{recovered - regressed}")

## 5.3 Two-Dimensional Scoring

Answer correctness is necessary but not sufficient. A system that produces the right answer through the wrong reasoning path is unreliable — it will break on the next similar question where luck does not hold.

For each agent loop result, score on two dimensions:

- **answer_correct** — does the final answer match the expected answer?
- **reasoning_correct** — did the agent select the right tool for the right reason?

The reasoning dimension requires human judgment. Look at the trace for each question and assess: was the tool selection appropriate? Were the query arguments reasonable? Did the agent use the tool result correctly?

> **Facilitator note:** This is a discussion exercise. Walk through 2-3 examples as a group. The point is not to score every question perfectly — it is to demonstrate that answer correctness alone is an incomplete evaluation.

In [ ]:
# Two-dimensional scoring
# answer_correct comes from the model judge in Section 4
# reasoning_correct is filled in by participants based on trace inspection
#
# Pre-filled with reasonable defaults based on tool selection patterns.
# Participants should review and adjust based on the traces.

scoring = []

for ar in agent_results:
    answer_correct = ar.get("agent_classification") == "pass"
    
    # Default reasoning assessment based on tool selection pattern
    # Participants should override these after inspecting traces
    tools_used = ar.get("tools_used", [])
    
    # A question that uses rag_retrieval for a rules question is reasoning-correct
    # A question that uses no_answer when corpus lacks info is reasoning-correct
    # A question that uses calculator for a math question is reasoning-correct
    reasoning_correct = len(tools_used) > 0  # default: correct if any tool was used
    
    scoring.append({
        "id": ar["id"],
        "question": ar["question"][:60],
        "answer_correct": answer_correct,
        "reasoning_correct": reasoning_correct,
        "tools_used": tools_used
    })

# Display the scoring table
print(f"{'ID':<6} {'Answer':<10} {'Reasoning':<12} {'Tools Used':<30} {'Question'}")
print("=" * 100)
for s in scoring:
    ans = "correct" if s["answer_correct"] else "WRONG"
    rsn = "correct" if s["reasoning_correct"] else "WRONG"
    tools = ", ".join(s["tools_used"]) if s["tools_used"] else "none"
    print(f"{s['id']:<6} {ans:<10} {rsn:<12} {tools:<30} {s['question']}")

## 5.4 The 2x2 Matrix

The two dimensions produce four categories. Each category points to a different kind of action.

In [ ]:
# Categorize each result into the 2x2 matrix
# Each quadrant tells you something different about system reliability

reliable = []        # correct answer + correct reasoning
lucky = []           # correct answer + wrong reasoning
corpus_gap = []      # wrong answer + correct reasoning
tool_problem = []    # wrong answer + wrong reasoning

for s in scoring:
    if s["answer_correct"] and s["reasoning_correct"]:
        reliable.append(s["id"])
    elif s["answer_correct"] and not s["reasoning_correct"]:
        lucky.append(s["id"])
    elif not s["answer_correct"] and s["reasoning_correct"]:
        corpus_gap.append(s["id"])
    else:
        tool_problem.append(s["id"])

print("2x2 Evaluation Matrix")
print("=" * 60)
print()
print(f"  Correct answer + Correct reasoning : RELIABLE")
print(f"    {reliable if reliable else '(none)'}")
print()
print(f"  Correct answer + Wrong reasoning   : LUCKY (not reliable)")
print(f"    {lucky if lucky else '(none)'}")
print()
print(f"  Wrong answer + Correct reasoning   : CORPUS GAP (fixable)")
print(f"    {corpus_gap if corpus_gap else '(none)'}")
print()
print(f"  Wrong answer + Wrong reasoning     : TOOL/DEFINITION PROBLEM")
print(f"    {tool_problem if tool_problem else '(none)'}")
print()
print("=" * 60)
print(f"Reliable: {len(reliable)}/10  |  Lucky: {len(lucky)}/10  |  "
      f"Corpus gap: {len(corpus_gap)}/10  |  Tool problem: {len(tool_problem)}/10")

> **Facilitator note:** The "lucky" quadrant is the most important one to discuss. A correct answer from wrong reasoning will fool a simple accuracy metric. It will not fool a user who asks a similar question tomorrow and gets a wrong answer. Reliability requires both dimensions to be correct.

---

## 5.5 Failure Analysis

For any question that is not in the "reliable" quadrant, the 2x2 matrix tells you where to look:

- **Lucky** — The tool description needs refinement. The model selected a tool that happened to produce useful context, but for the wrong reason. Fix the description.
- **Corpus gap** — The agent's reasoning was sound, but the corpus does not contain the answer. Fix the data, not the agent.
- **Tool problem** — Both the tool selection and the answer were wrong. Start by examining the trace: did the model misunderstand the tool description, or did the tool return unhelpful results?

Each failure maps to a specific layer. That is the value of two-dimensional scoring: it tells you *where* to intervene, not just *whether* to intervene.

In [ ]:
# For each non-reliable result, print the diagnosis
print("Failure Analysis")
print("=" * 60)

for s in scoring:
    if s["answer_correct"] and s["reasoning_correct"]:
        continue  # Skip reliable results
    
    # Find the full agent result for trace details
    ar = next(a for a in agent_results if a["id"] == s["id"])
    
    if s["answer_correct"] and not s["reasoning_correct"]:
        category = "LUCKY \u2014 correct answer, wrong reasoning"
        action = "Review and fix tool description"
    elif not s["answer_correct"] and s["reasoning_correct"]:
        category = "CORPUS GAP \u2014 wrong answer, correct reasoning"
        action = "Improve corpus coverage or chunking"
    else:
        category = "TOOL PROBLEM \u2014 wrong answer, wrong reasoning"
        action = "Examine trace and fix tool definition or dispatch"
    
    print(f"\n  {s['id']}: {category}")
    print(f"  Question : {ar['question']}")
    print(f"  Tools    : {', '.join(ar.get('tools_used', []))}")
    print(f"  Action   : {action}")
    if ar.get("judge_reason"):
        print(f"  Judge    : {ar['judge_reason']}")

## 5.6 Cumulative Results

The table below shows results across all evaluation approaches: passive RAG baseline from the Escalation Lab and the agent loop from this lab.

In [ ]:
# Build the cumulative comparison table
print("Cumulative Evaluation Results")
print("=" * 70)
print(f"{'ID':<6} {'Category':<22} {'Passive RAG':<14} {'Agent Loop'}")
print("-" * 70)

for br in baseline_results:
    ar = next((a for a in agent_results if a["id"] == br["id"]), None)
    
    passive = "pass" if br["classification"] == "pass" else "FAIL"
    agent = "pass" if (ar and ar.get("agent_classification") == "pass") else "FAIL"
    
    print(f"{br['id']:<6} {br.get('category', ''):<22} {passive:<14} {agent}")

passive_total = sum(1 for r in baseline_results if r["classification"] == "pass")
agent_total = sum(1 for r in agent_results if r.get("agent_classification") == "pass")

print("-" * 70)
print(f"{'Total':<6} {'':<22} {passive_total}/10{'':<9} {agent_total}/10")
print()
print(f"Net improvement from agent loop: +{agent_total - passive_total} questions")

> **FIELD TAKEAWAY**
>
> Accuracy alone is a dangerous metric for agentic systems. A system that gets the right answer through the wrong reasoning path is unreliable — it will fail unpredictably on similar questions. Two-dimensional evaluation (answer correctness + reasoning correctness) separates reliable results from lucky ones, and tells you exactly which layer to fix when something goes wrong: the tool description, the corpus, or the control structure.

## Pre-Built Output

If the cells above did not execute due to endpoint availability or time constraints, run the cell below to load pre-built results and continue the discussion.

This is expected behavior during a workshop. Use the pre-built outputs without apology and keep the discussion moving.

In [ ]:
USE_PREBUILT = False  # Set to True if live execution was not available

if USE_PREBUILT:
    import json
    
    with open("../prebuilt/eval_results.json", "r", encoding="utf-8") as f:
        baseline_data = json.load(f)
    baseline_results = baseline_data["results"]
    
    with open("../prebuilt/agent_loop_results.json", "r", encoding="utf-8") as f:
        agent_data = json.load(f)
    agent_results = agent_data["results"]
    
    print("Loaded pre-built evaluation outputs")
    print(f"  Baseline results : {len(baseline_results)}")
    print(f"  Agent results    : {len(agent_results)}")
    
    passive_total = sum(1 for r in baseline_results if r["classification"] == "pass")
    agent_total = sum(1 for r in agent_results if r.get("agent_classification") == "pass")
    print(f"\n  Passive RAG: {passive_total}/10")
    print(f"  Agent Loop : {agent_total}/10")
    print("\nReady to continue to Section 6.")
else:
    print("Using live results. Ready to continue to Section 6.")

---

## What Comes Next

Section 6 is a facilitated discussion. No new code. No live API calls. The results are in. The question now is: what do they mean for real engagements, and when is the added complexity of an agent loop justified?

Move to `06_Synthesis/06_Synthesis.ipynb`.